# RoBERTa-large-BNE + TF-IDF Multivista + MLP (Accuracy)

Notebook corregido para el problema de predicción de década.

Cambios clave:
- Usa `Wariano/roberta-large-bne`.
- Instala `tokenizers`, `sentencepiece` y dependencias necesarias.
- Usa **mean pooling** con `attention_mask`, no solo token CLS.
- Mantiene una rama TF-IDF multivista separada: `char + char_wb + word`.
- Fusiona ambos vectores en un MLP final.
- Métrica principal: **accuracy**.
- Early stopping: `patience = 2`.
- Incluye diagnóstico de etiquetas y tokenizer.
- Genera dos submissions:
  - modelo validado con split train/val/test
  - modelo final entrenado con todo `train.csv`

Archivos esperados:
- `train.csv` con columnas `text,decade`
- `eval.csv` con columnas `id,text`


In [ ]:
# =========================
# 0. Instalación
# =========================
# Ejecuta esta celda y REINICIA EL KERNEL antes de seguir.

%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

%pip install -U transformers accelerate sentencepiece protobuf tokenizers scikit-learn pandas numpy matplotlib tqdm scipy joblib


In [ ]:
# =========================
# 1. Imports
# =========================

import os
import gc
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, mean_absolute_error

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")


In [ ]:
# =========================
# 2. Configuración general / GPU
# =========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("ENTORNO")
print("=" * 70)
print("Python disponible en notebook")
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("CUDA PyTorch:", torch.version.cuda)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria total:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("=" * 70)


In [ ]:
# =========================
# 3. Limpieza inicial de memoria
# =========================

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("GPU asignada:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
        print("GPU reservada:", round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")

clean_memory()


In [ ]:
# =========================
# 4. Configuración del experimento
# =========================

TRAIN_PATH = "train.csv"
EVAL_PATH = "eval.csv"

TEXT_COL = "text"
LABEL_COL = "decade"

# Modelo RoBERTa no deprecado que te funcionó
ROBERTA_MODEL_NAME = "Wariano/roberta-large-bne"

# Longitud y batch. Si tienes GPU grande, puedes subir BATCH_SIZE.
MAX_LEN = 512
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2

# TF-IDF multivista
CHAR_NGRAM_RANGE = (2, 6)
CHAR_WB_NGRAM_RANGE = (3, 6)
WORD_NGRAM_RANGE = (1, 2)

CHAR_MAX_FEATURES = 180_000
CHAR_WB_MAX_FEATURES = 120_000
WORD_MAX_FEATURES = 80_000

MIN_DF = 2
MAX_DF = 1.0

# Dimensiones MLP
ROBERTA_PROJ_DIM = 512
TFIDF_HIDDEN = 768
TFIDF_PROJ_DIM = 512

FUSION_HIDDEN_1 = 1024
FUSION_HIDDEN_2 = 512
FUSION_HIDDEN_3 = 256

DROPOUT = 0.35
TFIDF_DROPOUT = 0.45

# Entrenamiento
EPOCHS = 30
PATIENCE = 2

LR_ROBERTA = 1e-5
LR_HEAD = 2e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
GRAD_CLIP = 1.0
WARMUP_RATIO = 0.08

CHECKPOINT_DIR = Path("checkpoints_roberta_tfidf_hybrid_accuracy")
CHECKPOINT_DIR.mkdir(exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_roberta_tfidf_hybrid.pt"
LAST_MODEL_PATH = CHECKPOINT_DIR / "last_roberta_tfidf_hybrid.pt"

SUBMISSION_VALID_PATH = "submission_roberta_tfidf_hybrid_validated.csv"
SUBMISSION_FULL_PATH = "submission_roberta_tfidf_hybrid_full_data.csv"

print("Modelo:", ROBERTA_MODEL_NAME)
print("Device:", DEVICE)
print("MAX_LEN:", MAX_LEN)
print("BATCH_SIZE:", BATCH_SIZE)
print("GRAD_ACCUM_STEPS:", GRAD_ACCUM_STEPS)
print("Patience:", PATIENCE)


In [ ]:
# =========================
# 5. Cargar dataset y mapeo de etiquetas
# =========================

df = pd.read_csv(TRAIN_PATH)

assert TEXT_COL in df.columns, f"Falta columna {TEXT_COL}"
assert LABEL_COL in df.columns, f"Falta columna {LABEL_COL}"

df = df[[TEXT_COL, LABEL_COL]].copy()
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)

decades = sorted(df[LABEL_COL].unique())
expected_decades = list(range(150, 189))

print("Décadas encontradas:", decades)
print("Número de clases:", len(decades))

assert decades == expected_decades, "Las décadas no son exactamente 150..188. Revisa el dataset."

decade_to_idx = {decade: idx for idx, decade in enumerate(decades)}
idx_to_decade = {idx: decade for decade, idx in decade_to_idx.items()}

df["label_idx"] = df[LABEL_COL].map(decade_to_idx)

NUM_CLASSES = len(decades)

print("Mapeo inicial:", list(decade_to_idx.items())[:5])
print("Mapeo final:", list(decade_to_idx.items())[-5:])
print("label_idx min/max:", df["label_idx"].min(), df["label_idx"].max())

assert df["label_idx"].min() == 0
assert df["label_idx"].max() == NUM_CLASSES - 1

display(df.head())
display(df[LABEL_COL].value_counts().sort_index())


In [ ]:
# =========================
# 6. Split train / val / test
# =========================

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label_idx"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_idx"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = part[LABEL_COL].value_counts().sort_index()
    print(name, "150:", counts.get(150, 0), "188:", counts.get(188, 0), "n_classes:", part[LABEL_COL].nunique())
    assert part[LABEL_COL].nunique() == NUM_CLASSES


In [ ]:
# =========================
# 7. Tokenizer RoBERTa
# =========================

tokenizer = AutoTokenizer.from_pretrained(
    ROBERTA_MODEL_NAME,
    use_fast=False
)

print("Tokenizer cargado")
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)

sample = train_df.iloc[0][TEXT_COL]
enc = tokenizer(
    sample,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="pt"
)

print("input_ids:", enc["input_ids"].shape)
print("attention_mask:", enc["attention_mask"].shape)
print("Tokens no padding:", enc["attention_mask"].sum().item())
print(tokenizer.decode(enc["input_ids"][0][:80], skip_special_tokens=False))


In [ ]:
# =========================
# 8. TF-IDF multivista
# =========================

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=CHAR_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    dtype=np.float32
)

char_wb_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=CHAR_WB_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=CHAR_WB_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    dtype=np.float32
)

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=WORD_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=WORD_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    token_pattern=r"(?u)\b\w+\b",
    dtype=np.float32
)

train_texts = train_df[TEXT_COL].tolist()
val_texts = val_df[TEXT_COL].tolist()
test_texts = test_df[TEXT_COL].tolist()

print("Ajustando TF-IDF char...")
X_train_char = char_vectorizer.fit_transform(train_texts)
X_val_char = char_vectorizer.transform(val_texts)
X_test_char = char_vectorizer.transform(test_texts)

print("Ajustando TF-IDF char_wb...")
X_train_char_wb = char_wb_vectorizer.fit_transform(train_texts)
X_val_char_wb = char_wb_vectorizer.transform(val_texts)
X_test_char_wb = char_wb_vectorizer.transform(test_texts)

print("Ajustando TF-IDF word...")
X_train_word = word_vectorizer.fit_transform(train_texts)
X_val_word = word_vectorizer.transform(val_texts)
X_test_word = word_vectorizer.transform(test_texts)

X_train = sp.hstack([X_train_char, X_train_char_wb, X_train_word], format="csr", dtype=np.float32)
X_val = sp.hstack([X_val_char, X_val_char_wb, X_val_word], format="csr", dtype=np.float32)
X_test = sp.hstack([X_test_char, X_test_char_wb, X_test_word], format="csr", dtype=np.float32)

y_train = train_df["label_idx"].values.astype(np.int64)
y_val = val_df["label_idx"].values.astype(np.int64)
y_test = test_df["label_idx"].values.astype(np.int64)

TFIDF_INPUT_DIM = X_train.shape[1]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("TFIDF_INPUT_DIM:", TFIDF_INPUT_DIM)


In [ ]:
# =========================
# 9. Datasets y DataLoaders
# =========================

class HybridDataset(Dataset):
    def __init__(self, dataframe, labels):
        self.texts = dataframe[TEXT_COL].astype(str).tolist()
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        encoded = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors=None
        )

        return {
            "idx": idx,
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
            "label": self.labels[idx],
        }

def hybrid_collate_fn(batch):
    return {
        "indices": torch.tensor([item["idx"] for item in batch], dtype=torch.long),
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
        "labels": torch.stack([item["label"] for item in batch]),
    }

train_dataset = HybridDataset(train_df, y_train)
val_dataset = HybridDataset(val_df, y_val)
test_dataset = HybridDataset(test_df, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=hybrid_collate_fn, num_workers=0, pin_memory=(DEVICE.type=="cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=hybrid_collate_fn, num_workers=0, pin_memory=(DEVICE.type=="cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=hybrid_collate_fn, num_workers=0, pin_memory=(DEVICE.type=="cuda"))

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))


In [ ]:
# =========================
# 10. Conversión sparse CSR a torch sparse
# =========================

def csr_batch_to_torch_sparse(X_csr_batch, device=DEVICE):
    X_coo = X_csr_batch.tocoo()

    indices = torch.tensor(
        np.vstack((X_coo.row, X_coo.col)),
        dtype=torch.long
    )

    values = torch.tensor(
        X_coo.data,
        dtype=torch.float32
    )

    shape = torch.Size(X_coo.shape)

    X_sparse = torch.sparse_coo_tensor(
        indices,
        values,
        shape,
        dtype=torch.float32
    )

    return X_sparse.coalesce().to(device)


In [ ]:
# =========================
# 11. Modelo híbrido RoBERTa mean pooling + TF-IDF + MLP
# CORRECCIÓN: la rama TF-IDF sparse se fuerza a float32 porque
# torch.sparse.mm no soporta Half en CUDA.
# =========================

class RobertaTfidfHybridClassifier(nn.Module):
    def __init__(
        self,
        roberta_model_name,
        tfidf_input_dim,
        num_classes,
        roberta_proj_dim=ROBERTA_PROJ_DIM,
        tfidf_hidden=TFIDF_HIDDEN,
        tfidf_proj_dim=TFIDF_PROJ_DIM,
        fusion_hidden_1=FUSION_HIDDEN_1,
        fusion_hidden_2=FUSION_HIDDEN_2,
        fusion_hidden_3=FUSION_HIDDEN_3,
        dropout=DROPOUT,
        tfidf_dropout=TFIDF_DROPOUT,
    ):
        super().__init__()

        self.roberta = AutoModel.from_pretrained(roberta_model_name)
        roberta_hidden = self.roberta.config.hidden_size

        self.roberta_projection = nn.Sequential(
            nn.Linear(roberta_hidden, roberta_proj_dim),
            nn.LayerNorm(roberta_proj_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Primera capa manual para input sparse TF-IDF.
        # Se define explícitamente en float32.
        self.tfidf_fc1_weight = nn.Parameter(
            torch.empty(tfidf_hidden, tfidf_input_dim, dtype=torch.float32)
        )
        self.tfidf_fc1_bias = nn.Parameter(
            torch.zeros(tfidf_hidden, dtype=torch.float32)
        )

        self.tfidf_block = nn.Sequential(
            nn.LayerNorm(tfidf_hidden),
            nn.GELU(),
            nn.Dropout(tfidf_dropout),
            nn.Linear(tfidf_hidden, tfidf_proj_dim),
            nn.LayerNorm(tfidf_proj_dim),
            nn.GELU(),
            nn.Dropout(tfidf_dropout)
        )

        combined_dim = roberta_proj_dim + tfidf_proj_dim

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, fusion_hidden_1),
            nn.LayerNorm(fusion_hidden_1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(fusion_hidden_1, fusion_hidden_2),
            nn.LayerNorm(fusion_hidden_2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(fusion_hidden_2, fusion_hidden_3),
            nn.LayerNorm(fusion_hidden_3),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(fusion_hidden_3, num_classes)
        )

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.tfidf_fc1_weight)
        nn.init.zeros_(self.tfidf_fc1_bias)

        # No reinicializamos RoBERTa. Sus pesos son preentrenados.
        for module in self.roberta_projection.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

        for module in self.tfidf_block.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

        for module in self.classifier.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counts

    def tfidf_sparse_projection(self, x_tfidf_sparse):
        """
        Proyección segura para TF-IDF sparse.
        Importante: torch.sparse.mm en CUDA no soporta float16/Half.
        Por eso desactivamos autocast y forzamos float32 en esta operación.
        """
        x_tfidf_sparse = x_tfidf_sparse.float()

        with torch.amp.autocast(
            device_type="cuda",
            enabled=False
        ):
            tfidf_vec = (
                torch.sparse.mm(
                    x_tfidf_sparse,
                    self.tfidf_fc1_weight.float().t()
                )
                + self.tfidf_fc1_bias.float()
            )

        return tfidf_vec

    def forward(self, input_ids, attention_mask, x_tfidf_sparse):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Mean pooling: más estable que CLS en texto ruidoso.
        roberta_vec = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        roberta_vec = self.roberta_projection(roberta_vec)

        # Rama TF-IDF sparse en float32 para evitar addmm_sparse_cuda Half.
        tfidf_vec = self.tfidf_sparse_projection(x_tfidf_sparse)
        tfidf_vec = self.tfidf_block(tfidf_vec)

        combined = torch.cat([roberta_vec.float(), tfidf_vec.float()], dim=1)
        logits = self.classifier(combined)

        return logits


model = RobertaTfidfHybridClassifier(
    roberta_model_name=ROBERTA_MODEL_NAME,
    tfidf_input_dim=TFIDF_INPUT_DIM,
    num_classes=NUM_CLASSES
).to(DEVICE)

try:
    model.roberta.gradient_checkpointing_enable()
    print("Gradient checkpointing activado.")
except Exception as e:
    print("No se pudo activar gradient checkpointing:", e)

if hasattr(model.roberta.config, "use_cache"):
    model.roberta.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


In [ ]:
# =========================
# 12. Diagnóstico rápido de batch
# =========================

batch = next(iter(train_loader))

print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["labels"][:20])
print("tokens no padding:", batch["attention_mask"].sum(dim=1)[:20])
print("decode ejemplo:")
print(tokenizer.decode(batch["input_ids"][0][:120], skip_special_tokens=False))

batch_indices = batch["indices"].numpy()
X_batch = csr_batch_to_torch_sparse(X_train[batch_indices], device=DEVICE)

model.eval()
with torch.no_grad():
    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    logits = model(input_ids, attention_mask, X_batch)

print("logits:", logits.shape)
assert logits.shape == (batch["labels"].shape[0], NUM_CLASSES)
print("Forward correcto.")
clean_memory()


In [ ]:
# =========================
# 13. Métricas
# =========================

idx_to_decade_arr = np.array([idx_to_decade[i] for i in range(NUM_CLASSES)])

def compute_metrics_from_preds(preds, labels):
    acc = np.mean(preds == labels)
    pred_decades = idx_to_decade_arr[preds]
    true_decades = idx_to_decade_arr[labels]
    abs_err = np.abs(pred_decades - true_decades)
    return {
        "acc": float(acc),
        "mae": float(abs_err.mean()),
        "acc_pm1": float(np.mean(abs_err <= 1)),
        "acc_pm2": float(np.mean(abs_err <= 2)),
    }

def compute_metrics_from_logits(logits, labels):
    preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
    labels_np = labels.detach().cpu().numpy()
    return compute_metrics_from_preds(preds, labels_np)


In [ ]:
# =========================
# 14. Optimizer, scheduler, loss
# =========================

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

roberta_params = []
head_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if name.startswith("roberta."):
        roberta_params.append(param)
    else:
        head_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": roberta_params, "lr": LR_ROBERTA, "weight_decay": WEIGHT_DECAY},
        {"params": head_params, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
    ]
)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_training_steps = num_update_steps_per_epoch * EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

scaler = torch.amp.GradScaler(
    device="cuda",
    enabled=(DEVICE.type == "cuda")
)

print("RoBERTa params:", sum(p.numel() for p in roberta_params))
print("Head params:", sum(p.numel() for p in head_params))
print("Total steps:", total_training_steps)
print("Warmup:", warmup_steps)


In [ ]:
# =========================
# 15. Training / evaluation
# =========================

def train_one_epoch(model, loader, X_matrix):
    model.train()

    total_loss = 0.0
    total_examples = 0

    all_logits = []
    all_labels = []

    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(loader, desc="Training", leave=False)

    for step, batch in enumerate(progress, start=1):
        batch_indices = batch["indices"].numpy()

        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        X_batch = csr_batch_to_torch_sparse(
            X_matrix[batch_indices],
            device=DEVICE
        )

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(input_ids, attention_mask, X_batch)
            loss = criterion(logits, labels)
            loss_for_backward = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss_for_backward).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        all_logits.append(logits.detach().float().cpu())
        all_labels.append(labels.detach().cpu())

        progress.set_postfix({
            "loss": total_loss / total_examples,
            "lr_r": f"{optimizer.param_groups[0]['lr']:.1e}",
            "lr_h": f"{optimizer.param_groups[1]['lr']:.1e}",
        })

        del X_batch, input_ids, attention_mask, labels, logits, loss

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    metrics = compute_metrics_from_logits(all_logits, all_labels)
    metrics["loss"] = total_loss / total_examples
    return metrics


@torch.no_grad()
def evaluate(model, loader, X_matrix, desc="Evaluating"):
    model.eval()

    total_loss = 0.0
    total_examples = 0

    all_logits = []
    all_labels = []

    progress = tqdm(loader, desc=desc, leave=False)

    for batch in progress:
        batch_indices = batch["indices"].numpy()

        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        X_batch = csr_batch_to_torch_sparse(
            X_matrix[batch_indices],
            device=DEVICE
        )

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(input_ids, attention_mask, X_batch)
            loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        all_logits.append(logits.detach().float().cpu())
        all_labels.append(labels.detach().cpu())

        del X_batch, input_ids, attention_mask, labels, logits, loss

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    metrics = compute_metrics_from_logits(all_logits, all_labels)
    metrics["loss"] = total_loss / total_examples
    return metrics


In [ ]:
# =========================
# 16. Checkpoint helpers
# =========================

def save_checkpoint(path, epoch, best_val_acc, history, extra=None):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_acc": best_val_acc,
        "history": history,
        "vectorizers": {
            "char": char_vectorizer,
            "char_wb": char_wb_vectorizer,
            "word": word_vectorizer,
        },
        "decade_to_idx": decade_to_idx,
        "idx_to_decade": idx_to_decade,
        "decades": decades,
        "config": {
            "ROBERTA_MODEL_NAME": ROBERTA_MODEL_NAME,
            "MAX_LEN": MAX_LEN,
            "TFIDF_INPUT_DIM": TFIDF_INPUT_DIM,
            "NUM_CLASSES": NUM_CLASSES,
            "ROBERTA_PROJ_DIM": ROBERTA_PROJ_DIM,
            "TFIDF_HIDDEN": TFIDF_HIDDEN,
            "TFIDF_PROJ_DIM": TFIDF_PROJ_DIM,
            "FUSION_HIDDEN_1": FUSION_HIDDEN_1,
            "FUSION_HIDDEN_2": FUSION_HIDDEN_2,
            "FUSION_HIDDEN_3": FUSION_HIDDEN_3,
            "DROPOUT": DROPOUT,
            "TFIDF_DROPOUT": TFIDF_DROPOUT,
        },
        "extra": extra or {},
    }, path)


def load_checkpoint_cpu(path):
    return torch.load(path, map_location="cpu", weights_only=False)


In [ ]:
# =========================
# 17. Loop principal
# =========================

history = []
best_val_acc = -1.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 80)

    train_metrics = train_one_epoch(model, train_loader, X_train)
    val_metrics = evaluate(model, val_loader, X_val, desc="Validation")

    row = {
        "epoch": epoch,
        "lr_roberta": optimizer.param_groups[0]["lr"],
        "lr_head": optimizer.param_groups[1]["lr"],

        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_mae": train_metrics["mae"],
        "train_pm1": train_metrics["acc_pm1"],
        "train_pm2": train_metrics["acc_pm2"],

        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_mae": val_metrics["mae"],
        "val_pm1": val_metrics["acc_pm1"],
        "val_pm2": val_metrics["acc_pm2"],
    }

    history.append(row)

    print(
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Train MAE: {row['train_mae']:.3f}"
    )

    print(
        f"Val loss:   {row['val_loss']:.4f} | "
        f"Val acc:   {row['val_acc']:.4f} | "
        f"Val MAE:   {row['val_mae']:.3f} | "
        f"Val ±1: {row['val_pm1']:.4f}"
    )

    save_checkpoint(LAST_MODEL_PATH, epoch, best_val_acc, history, {"type": "last"})

    if val_metrics["acc"] > best_val_acc:
        best_val_acc = val_metrics["acc"]
        best_epoch = epoch
        epochs_without_improvement = 0

        save_checkpoint(BEST_MODEL_PATH, epoch, best_val_acc, history, {"type": "best"})
        print("Nuevo mejor checkpoint guardado:", BEST_MODEL_PATH)
    else:
        epochs_without_improvement += 1
        print("Sin mejora:", epochs_without_improvement)

    clean_memory()

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping activado.")
        break

print("\nEntrenamiento terminado.")
print("Mejor época:", best_epoch)
print("Mejor val accuracy:", best_val_acc)


In [ ]:
# =========================
# 18. Curvas
# =========================

history_df = pd.DataFrame(history)
display(history_df)

plt.figure(figsize=(10,4))
plt.plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
plt.plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10,4))
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# =========================
# 19. Evaluación test + matriz de confusión
# =========================

checkpoint = load_checkpoint_cpu(BEST_MODEL_PATH)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Checkpoint:", BEST_MODEL_PATH)
print("Época:", checkpoint["epoch"])
print("Best val acc:", checkpoint["best_val_acc"])

test_metrics = evaluate(model, test_loader, X_test, desc="Test")

print("=" * 70)
print("TEST")
print("=" * 70)
print(test_metrics)

@torch.no_grad()
def get_predictions(model, loader, X_matrix, desc="Predictions"):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in tqdm(loader, desc=desc):
        batch_indices = batch["indices"].numpy()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        X_batch = csr_batch_to_torch_sparse(X_matrix[batch_indices], device=DEVICE)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
            logits = model(input_ids, attention_mask, X_batch)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

        del X_batch, input_ids, attention_mask, labels, logits, preds

    return np.array(all_preds), np.array(all_labels)

test_preds, test_labels = get_predictions(model, test_loader, X_test, desc="Test preds")

cm = confusion_matrix(test_labels, test_preds, labels=list(range(NUM_CLASSES)))

cm_df = pd.DataFrame(
    cm,
    index=[idx_to_decade[i] for i in range(NUM_CLASSES)],
    columns=[idx_to_decade[i] for i in range(NUM_CLASSES)]
)

display(cm_df)

plt.figure(figsize=(15, 12))
plt.imshow(cm, aspect="auto")
plt.title("Matriz de confusión sin normalizar")
plt.xlabel("Década predicha")
plt.ylabel("Década real")
plt.xticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)], rotation=90)
plt.yticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)])
plt.colorbar(label="Conteo")
plt.tight_layout()
plt.show()

print("Predicciones por clase:")
pred_counts = pd.Series([idx_to_decade[int(i)] for i in test_preds]).value_counts().sort_index()
display(pred_counts)
print("Predicciones 150:", pred_counts.get(150, 0))
print("Predicciones 188:", pred_counts.get(188, 0))


In [ ]:
# =========================
# 20. Submission del modelo validado
# =========================

eval_df = pd.read_csv(EVAL_PATH)

assert "id" in eval_df.columns
assert "text" in eval_df.columns

eval_df["text"] = eval_df["text"].fillna("").astype(str)
eval_texts = eval_df["text"].tolist()

X_eval_char = char_vectorizer.transform(eval_texts)
X_eval_char_wb = char_wb_vectorizer.transform(eval_texts)
X_eval_word = word_vectorizer.transform(eval_texts)

X_eval = sp.hstack([X_eval_char, X_eval_char_wb, X_eval_word], format="csr", dtype=np.float32)

class EvalHybridDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe[TEXT_COL].astype(str).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoded = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors=None
        )
        return {
            "idx": idx,
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
        }

def eval_hybrid_collate_fn(batch):
    return {
        "indices": torch.tensor([item["idx"] for item in batch], dtype=torch.long),
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
    }

eval_dataset = EvalHybridDataset(eval_df)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=eval_hybrid_collate_fn, num_workers=0)

@torch.no_grad()
def predict_eval(model, loader, X_matrix):
    model.eval()
    all_preds = []

    for batch in tqdm(loader, desc="Predicting eval"):
        batch_indices = batch["indices"].numpy()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        X_batch = csr_batch_to_torch_sparse(X_matrix[batch_indices], device=DEVICE)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
            logits = model(input_ids, attention_mask, X_batch)

        preds = torch.argmax(logits, dim=1).detach().cpu().numpy().tolist()
        all_preds.extend(preds)

        del X_batch, input_ids, attention_mask, logits

    return np.array(all_preds)

eval_pred_idx = predict_eval(model, eval_loader, X_eval)
answers = [idx_to_decade[int(i)] for i in eval_pred_idx]

submission = pd.DataFrame({
    "id": eval_df["id"].values,
    "answer": answers
})

submission.to_csv(SUBMISSION_VALID_PATH, index=False)

print("Submission guardada:", SUBMISSION_VALID_PATH)
display(submission.head())
display(submission["answer"].value_counts().sort_index())
print("Predicciones 150:", (submission["answer"] == 150).sum())
print("Predicciones 188:", (submission["answer"] == 188).sum())


In [ ]:
# =========================
# 21. Entrenamiento final con todo train.csv
# =========================

FINAL_EPOCHS = checkpoint["epoch"]

FINAL_CHECKPOINT_DIR = Path("checkpoints_roberta_tfidf_hybrid_full_data")
FINAL_CHECKPOINT_DIR.mkdir(exist_ok=True)
FINAL_MODEL_PATH = FINAL_CHECKPOINT_DIR / "final_roberta_tfidf_hybrid_full_data.pt"

print("FINAL_EPOCHS:", FINAL_EPOCHS)

full_df = df.copy().reset_index(drop=True)
full_texts = full_df[TEXT_COL].astype(str).tolist()
y_full = full_df["label_idx"].values.astype(np.int64)

final_char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=CHAR_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    dtype=np.float32
)

final_char_wb_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=CHAR_WB_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=CHAR_WB_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    dtype=np.float32
)

final_word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=WORD_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=WORD_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    token_pattern=r"(?u)\b\w+\b",
    dtype=np.float32
)

print("Ajustando TF-IDF final...")
X_full_char = final_char_vectorizer.fit_transform(full_texts)
X_full_char_wb = final_char_wb_vectorizer.fit_transform(full_texts)
X_full_word = final_word_vectorizer.fit_transform(full_texts)
X_full = sp.hstack([X_full_char, X_full_char_wb, X_full_word], format="csr", dtype=np.float32)

FINAL_TFIDF_INPUT_DIM = X_full.shape[1]
print("X_full:", X_full.shape)

final_dataset = HybridDataset(full_df, y_full)
final_loader = DataLoader(final_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=hybrid_collate_fn, num_workers=0, pin_memory=(DEVICE.type=="cuda"))

# Liberar modelo anterior
model.to("cpu")
del model
clean_memory()

final_model = RobertaTfidfHybridClassifier(
    roberta_model_name=ROBERTA_MODEL_NAME,
    tfidf_input_dim=FINAL_TFIDF_INPUT_DIM,
    num_classes=NUM_CLASSES
).to(DEVICE)

try:
    final_model.roberta.gradient_checkpointing_enable()
except Exception:
    pass

if hasattr(final_model.roberta.config, "use_cache"):
    final_model.roberta.config.use_cache = False

final_roberta_params = []
final_head_params = []

for name, param in final_model.named_parameters():
    if not param.requires_grad:
        continue
    if name.startswith("roberta."):
        final_roberta_params.append(param)
    else:
        final_head_params.append(param)

final_optimizer = torch.optim.AdamW(
    [
        {"params": final_roberta_params, "lr": LR_ROBERTA, "weight_decay": WEIGHT_DECAY},
        {"params": final_head_params, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
    ]
)

final_num_update_steps_per_epoch = math.ceil(len(final_loader) / GRAD_ACCUM_STEPS)
final_total_steps = final_num_update_steps_per_epoch * FINAL_EPOCHS
final_warmup_steps = int(final_total_steps * WARMUP_RATIO)

final_scheduler = get_linear_schedule_with_warmup(
    final_optimizer,
    num_warmup_steps=final_warmup_steps,
    num_training_steps=final_total_steps
)

final_scaler = torch.amp.GradScaler(device="cuda", enabled=(DEVICE.type=="cuda"))
final_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

print("Modelo final listo.")


In [ ]:
# =========================
# 22. Loop final con todo train.csv
# =========================

def train_final_one_epoch(model, loader, X_matrix):
    model.train()

    total_loss = 0.0
    total_examples = 0

    all_logits = []
    all_labels = []

    final_optimizer.zero_grad(set_to_none=True)

    progress = tqdm(loader, desc="Final training", leave=False)

    for step, batch in enumerate(progress, start=1):
        batch_indices = batch["indices"].numpy()

        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        X_batch = csr_batch_to_torch_sparse(X_matrix[batch_indices], device=DEVICE)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
            logits = model(input_ids, attention_mask, X_batch)
            loss = final_criterion(logits, labels)
            loss_for_backward = loss / GRAD_ACCUM_STEPS

        final_scaler.scale(loss_for_backward).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(loader):
            final_scaler.unscale_(final_optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            final_scaler.step(final_optimizer)
            final_scaler.update()

            final_optimizer.zero_grad(set_to_none=True)
            final_scheduler.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        all_logits.append(logits.detach().float().cpu())
        all_labels.append(labels.detach().cpu())

        progress.set_postfix({"loss": total_loss / total_examples})

        del X_batch, input_ids, attention_mask, labels, logits, loss

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    metrics = compute_metrics_from_logits(all_logits, all_labels)
    metrics["loss"] = total_loss / total_examples

    return metrics


final_history = []

for epoch in range(1, FINAL_EPOCHS + 1):
    print(f"\nFinal epoch {epoch}/{FINAL_EPOCHS}")
    print("-" * 80)

    metrics = train_final_one_epoch(final_model, final_loader, X_full)

    row = {
        "epoch": epoch,
        "loss": metrics["loss"],
        "acc": metrics["acc"],
        "mae": metrics["mae"],
        "pm1": metrics["acc_pm1"],
        "pm2": metrics["acc_pm2"],
    }

    final_history.append(row)

    print(row)

    torch.save({
        "epoch": epoch,
        "model_state_dict": final_model.state_dict(),
        "history": final_history,
        "vectorizers": {
            "char": final_char_vectorizer,
            "char_wb": final_char_wb_vectorizer,
            "word": final_word_vectorizer,
        },
        "decade_to_idx": decade_to_idx,
        "idx_to_decade": idx_to_decade,
        "decades": decades,
        "config": {
            "ROBERTA_MODEL_NAME": ROBERTA_MODEL_NAME,
            "MAX_LEN": MAX_LEN,
            "TFIDF_INPUT_DIM": FINAL_TFIDF_INPUT_DIM,
            "NUM_CLASSES": NUM_CLASSES,
        }
    }, FINAL_CHECKPOINT_DIR / f"final_epoch_{epoch:02d}.pt")

    clean_memory()

torch.save({
    "epoch": FINAL_EPOCHS,
    "model_state_dict": final_model.state_dict(),
    "history": final_history,
    "vectorizers": {
        "char": final_char_vectorizer,
        "char_wb": final_char_wb_vectorizer,
        "word": final_word_vectorizer,
    },
    "decade_to_idx": decade_to_idx,
    "idx_to_decade": idx_to_decade,
    "decades": decades,
    "config": {
        "ROBERTA_MODEL_NAME": ROBERTA_MODEL_NAME,
        "MAX_LEN": MAX_LEN,
        "TFIDF_INPUT_DIM": FINAL_TFIDF_INPUT_DIM,
        "NUM_CLASSES": NUM_CLASSES,
    }
}, FINAL_MODEL_PATH)

print("Modelo final guardado:", FINAL_MODEL_PATH)


In [ ]:
# =========================
# 23. Submission final full-data
# =========================

final_char_vectorizer = final_char_vectorizer
final_char_wb_vectorizer = final_char_wb_vectorizer
final_word_vectorizer = final_word_vectorizer

eval_df = pd.read_csv(EVAL_PATH)
eval_df["text"] = eval_df["text"].fillna("").astype(str)
eval_texts = eval_df["text"].tolist()

X_eval_char = final_char_vectorizer.transform(eval_texts)
X_eval_char_wb = final_char_wb_vectorizer.transform(eval_texts)
X_eval_word = final_word_vectorizer.transform(eval_texts)

X_eval_final = sp.hstack([X_eval_char, X_eval_char_wb, X_eval_word], format="csr", dtype=np.float32)

eval_dataset = EvalHybridDataset(eval_df)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=eval_hybrid_collate_fn, num_workers=0)

eval_pred_idx = predict_eval(final_model, eval_loader, X_eval_final)
answers = [idx_to_decade[int(i)] for i in eval_pred_idx]

submission_full = pd.DataFrame({
    "id": eval_df["id"].values,
    "answer": answers
})

submission_full.to_csv(SUBMISSION_FULL_PATH, index=False)

print("Submission full-data guardada:", SUBMISSION_FULL_PATH)
display(submission_full.head())
display(submission_full["answer"].value_counts().sort_index())
print("Predicciones 150:", (submission_full["answer"] == 150).sum())
print("Predicciones 188:", (submission_full["answer"] == 188).sum())
